# Jev Fundamentals: Building a Customer Support Ticket Classifier

In this notebook, we will learn how to use **Jev**, TypeSafe AI's
"System One" model, through LangChain.

Instead of generating text like a traditional LLM, Jev answers
structured questions about our input.

We will build a small customer-support classification system that
can answer questions such as:

- Is this ticket urgent?
- Which department should handle it?
- How frustrated is the customer?

By the end of this notebook, we will have a reusable Python function
that can classify support tickets using a single Jev API call.

## 1. What is Jev?

Traditional LLMs are designed primarily to generate text.

For example:

    "Is this customer issue urgent?"

An LLM might respond:

    "Yes, this appears to be urgent because the customer
     has been unable to make payments for three days."

But many AI applications don't need a paragraph of text.

They simply need a decision:

    urgent = YES

This is where Jev is useful.

Jev is designed to answer typed questions such as:

- Yes / No
- Choose one option
- Give a score

It also provides probabilities that can help an application
decide how confident the classification is.

The important mental model is:

    LLM → generates language

    Jev → makes fast structured decisions

In [9]:
import os 
from dotenv import load_dotenv
from langchain_typesafe import (
    TypeSafeClassifier,
    Noul,
    Choice,
    Score
)


In [10]:
classifier = TypeSafeClassifier()

In [11]:
ticket = """
My payment has failed three times today.
I urgently need to complete this transaction.
Please help me immediately.
"""

response = classifier.invoke({
    "state": ticket,
    "questions": {
        "urgent": Noul(
            instructions="Does this customer message express urgency?"
        )
    }
})

print(response)

model='jev-1.13.0' answers={'urgent': NoulAnswer(type='noul', noul=0.99)} usage=Usage(input_tokens=295, output_tokens=20) request_id='req_01a0f084b05c7db4935162ad8211eada'


##  Asking multiple questions in one call

Instead of making three separate API calls:

    Call 1 → Is it urgent?
    Call 2 → Is it a billing issue?
    Call 3 → Does the customer need human help?

we can ask multiple questions in the same classification request.

This is useful because all questions are evaluated against
the same input.

In [12]:
ticket = """
I have been trying to process my payment for three days.
Every attempt fails and I cannot complete my purchase.
Please get someone from your billing team to help me.
"""

response = classifier.invoke({
    "state": ticket,
    
    "questions": {
        "urgent": Noul(
            instructions="Does this customer message express urgency?"
        ),
        
        "needs_human": Noul(
            instructions="Does the customer appear to need human support?"
        ),
        
        "customer_impact": Noul(
            instructions="Is the customer currently unable to complete an important action?"
        )
    }
})

print("Urgent:", response.nouls["urgent"].noul)

print("Needs human:",
      response.nouls["needs_human"].noul)

print("Customer impact:",
      response.nouls["customer_impact"].noul)

Urgent: 0.87
Needs human: 0.96
Customer impact: 0.97


## Using Choice questions

Sometimes YES/NO isn't enough.

For example, our support team might have:

    Billing
    Technical
    Account
    General

We want Jev to choose the most appropriate department.

For this we use Choice.

In [13]:
ticket = """
I have been charged twice for the same subscription.
Please refund the duplicate payment.
"""

response = classifier.invoke({
    "state": ticket,
    
    "questions": {
        "department": Choice(
            instructions="Which department should handle this support ticket?",
            
            criteria={
                "billing": "Payment, invoice, subscription or refund issues",
                "technical": "Product bugs, errors or technical problems",
                "account": "Login, account access or profile issues",
                "general": "Questions that do not clearly belong to another department"
            }
        )
    }
})

department = response.choices["department"]

print(department)


print("Selected department:")
print(department.choice)

type='choice' choice='billing' probabilities={'technical': 0.0, 'general': 0.0, 'billing': 1.0, 'account': 0.0} confidence=1.0
Selected department:
billing


### Why do we provide criteria?

Instead of only giving Jev:

    billing
    technical
    account
    general

we explain what each option means.

This gives Jev context for making the classification.

For example:

    billing → payment, invoice, subscription or refund issues

This makes the question much more explicit and easier to maintain.

##  Measuring customer frustration

Now let's use a Score question.

Suppose we want to understand how frustrated the customer appears.

We'll define three levels:

    calm
    frustrated
    angry

Jev will estimate where the message falls on this scale.

In [14]:
ticket = """
This is absolutely ridiculous!

I have contacted support four times and nobody has fixed
the problem. I have been charged but still don't have access.

I am extremely frustrated with this service.
"""

response = classifier.invoke({
    "state": ticket,
    
    "questions": {
        "frustration": Score(
            instructions="How frustrated does the customer appear?",
            criteria=[
                "calm",
                "frustrated",
                "angry"
            ]
        )
    }
})

frustration = response.scores["frustration"]

print(frustration)

print("Frustration score:")
print(frustration.score)

type='score' score=1.93 legend={0: 'calm', 1: 'frustrated', 2: 'angry'} probabilities={0: 0.0, 1: 0.07, 2: 0.93} confidence=0.9
Frustration score:
1.93


##  Building our complete support classifier

Our final classifier will answer three questions:

1. Is the ticket urgent?
2. Which department should handle it?
3. How frustrated is the customer?

All three questions will be sent in one Jev classification request.

In [15]:
def classify_support_ticket(ticket):
    
    response = classifier.invoke({
        "state": ticket,
        
        "questions": {
            "urgent": Noul(
                instructions="Does this customer issue require urgent attention?"
            ),
            
            "department": Choice(
                instructions="Which department should handle this ticket?",
                
                criteria={
                    "billing": "Payments, invoices, subscriptions or refunds",
                    "technical": "Bugs, errors or technical problems",
                    "account": "Login, account access or profile issues",
                    "general": "General questions or requests"
                }
            ),
            
            "frustration": Score(
                instructions="How frustrated does the customer appear?",
                criteria=[
                    "calm",
                    "frustrated",
                    "angry"
                ]
            )
        }
    })
    
    return response

ticket = """
I have been charged twice for my subscription.
I have already contacted support once but the issue
has not been resolved.

Please fix this as soon as possible.
"""

result = classify_support_ticket(ticket)

print("Urgent:")
print(result.nouls["urgent"].noul)

print("\nDepartment:")
print(result.choices["department"].choice)

print("\nFrustration:")
print(result.scores["frustration"].score)

Urgent:
0.84

Department:
billing

Frustration:
1.0


##### Running Multiple Tickets using loop

In [16]:
tickets = [
    """
    I cannot login to my account. I keep getting an invalid password
    error even though I know my password is correct.
    """,
    
    """
    My credit card was charged twice for the same monthly subscription.
    Please refund the duplicate payment.
    """,
    
    """
    The application crashes every time I try to upload a CSV file.
    This is blocking our team from completing today's work.
    """,
    
    """
    Can you tell me what plans you offer for small businesses?
    I would like some information before deciding.
    """
]
for i, ticket in enumerate(tickets, start=1):
    
    result = classify_support_ticket(ticket)
    
    print(f"\n{'=' * 50}")
    print(f"Ticket {i}")
    print(f"{'=' * 50}")
    
    print("Urgent:",
          result.nouls["urgent"].noul)
    
    print("Department:",
          result.choices["department"].choice)
    
    print("Frustration:",
          result.scores["frustration"].score)


Ticket 1
Urgent: 0.41
Department: account
Frustration: 0.89

Ticket 2
Urgent: 0.4
Department: billing
Frustration: 0.18

Ticket 3
Urgent: 0.93
Department: technical
Frustration: 1.01

Ticket 4
Urgent: 0.08
Department: general
Frustration: 0.0
